# AWS Lambda Functions with Boto3

Region: ap-south-1

In [ ]:
# create the iam and lambda clients
import boto3
import json
import time
import uuid
import zipfile
import io

REGION = "ap-south-1"
iam = boto3.client("iam")
lambda_client = boto3.client("lambda", region_name=REGION)

role_name = f"exam-lambda-role-{uuid.uuid4().hex[:8]}"
function_name = f"exam-lambda-{uuid.uuid4().hex[:8]}"
runtime = "python3.11"
handler = "lambda_function.lambda_handler"

assume_role_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

lambda_code = [
    "def lambda_handler(event, context):\n",
    "    return {\"statusCode\": 200, \"body\": \"ok\"}\n",
]

buffer = io.BytesIO()
with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.writestr("lambda_function.py", "".join(lambda_code))
zip_bytes = buffer.getvalue()

In [ ]:
# create the iam role
response = iam.create_role(
    RoleName=role_name,
    AssumeRolePolicyDocument=json.dumps(assume_role_policy),
)
print(response)

response = iam.attach_role_policy(
    RoleName=role_name,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)
print(response)

time.sleep(10)

In [ ]:
# create the lambda function
role_arn = iam.get_role(RoleName=role_name)["Role"]["Arn"]

response = lambda_client.create_function(
    FunctionName=function_name,
    Runtime=runtime,
    Role=role_arn,
    Handler=handler,
    Code={"ZipFile": zip_bytes},
    Timeout=10,
)
print(response)

In [ ]:
# invoke the function
response = lambda_client.invoke(
    FunctionName=function_name,
    Payload=json.dumps({"ping": "pong"}).encode("utf-8"),
)
print({k: v for k, v in response.items() if k != "Payload"})
print(response["Payload"].read().decode("utf-8"))

In [ ]:
# get the function configuration
response = lambda_client.get_function(
    FunctionName=function_name,
)
print(response)

In [ ]:
# delete the function and role
response = lambda_client.delete_function(
    FunctionName=function_name,
)
print(response)

response = iam.detach_role_policy(
    RoleName=role_name,
    PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
)
print(response)

response = iam.delete_role(
    RoleName=role_name,
)
print(response)